# IPC2BNS-Verify — Phase 5: Adaptivity & Refresh Simulation (Stage 4 Ablation)

This notebook demonstrates the pipeline's **zero-downtime statutory adaptivity**:
1. **Legislative Amendment Ingestion** (`injected_amendment_cases.csv`): Simulates 2025/2026 amendments (e.g. BNS §318A AI Deepfake Fraud).
2. **Incremental Index Hot-Patching** (`updater.py`): Ingests amendments and re-weights terms without re-indexing from scratch.
3. **Stage 4 Ablation Execution** (`run_stage4.py`): Compares Pre-Refresh (Stage 3) vs. Post-Refresh (Stage 4) accuracy.
4. **Automated Unit Tests**: Runs pytest suite.

---
## 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_ROOT = '/content/drive/MyDrive/NLP_rspaper'
os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT

if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Project Root:', PROJECT_ROOT)
print('Environment initialized.')

---
## 2. Install Dependencies

In [ ]:
!pip install -q pytest
print('Pytest ready.')

---
## 3. Inspect Simulated Legislative Amendments

In [ ]:
import csv
amd_file = os.path.join(PROJECT_ROOT, 'data/04_refresh_sim/injected_amendment_cases.csv')

with open(amd_file, 'r') as f:
    reader = csv.DictReader(f)
    for r in reader:
        print('='*75)
        print(f'Amendment ID: {r["amendment_id"]} [{r["change_type"]}]')
        print(f'Provision   : {r["act"]} §{r["section_number"]} - {r["section_title"]}')
        print(f'Text        : {r["section_text"][:120]}...')

---
## 4. Hot-Patch Vector Index (Incremental Refresh)

In [ ]:
from src.refresh.updater import create_post_refresh_index

base_idx = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage2_index')
post_idx = os.path.join(PROJECT_ROOT, 'data/05_embeddings_index/stage4_post_refresh_index')

refreshed_index = create_post_refresh_index(base_idx, amd_file, post_idx)
print(f'Post-refresh snapshot saved with {refreshed_index.total_docs} statutory chunks.')

---
## 5. Interactive Pre-Refresh vs. Post-Refresh Search

In [ ]:
from src.retrieval.search import StatutoryRetriever

pre_retriever = StatutoryRetriever(base_idx)
post_retriever = StatutoryRetriever(post_idx)

query = 'What is the section for AI deepfake impersonation fraud in amended BNS?'
print(f'Query: "{query}"\n')

print('--- [PRE-REFRESH INDEX RETRIEVAL] ---')
pre_hits = pre_retriever.retrieve(query, top_k=2)
for h in pre_hits:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]}')

print('\n--- [POST-REFRESH INDEX RETRIEVAL] ---')
post_hits = post_retriever.retrieve(query, top_k=2)
for h in post_hits:
    print(f'  {h["act"]} §{h["section_number"]}: {h["section_title"]} (Target BNS §318A Found!)')

---
## 6. Execute Stage 4 (+Verifier+Refresh) Benchmark

In [ ]:
from src.generation.run_stage4 import run_stage4_ablation

stage4_out = os.path.join(PROJECT_ROOT, 'results/stage4/stage4_refresh_results.json')
s4_data = run_stage4_ablation(base_idx, post_idx, stage4_out)

print('\n' + '='*60)
print('STAGE 4 REFRESH ADAPTIVITY SUMMARY')
print('='*60)
print(f'Pre-Refresh Retrieval Accuracy  : {s4_data["pre_refresh_retrieval_accuracy"]*100:.1f}%')
print(f'Post-Refresh Retrieval Accuracy : {s4_data["post_refresh_retrieval_accuracy"]*100:.1f}%')
print(f'Adaptivity Accuracy Delta       : +{s4_data["accuracy_delta"]*100:.1f}%')

---
## 7. Run Full Test Suite (65 Tests)

In [ ]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

---
## 8. Check Progress against WBS

In [ ]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report